## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value!

In the folder `me` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours!

I've also made a file called `summary.txt`

We're not going to use Tools just yet - we're going to add the tool tomorrow.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. You can get guides to these packages by asking 
            ChatGPT or Claude, and you find all open-source packages on the repository <a href="https://pypi.org">https://pypi.org</a>.
            </span>
        </td>
    </tr>
</table>

In [2]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr
import json

In [3]:
load_dotenv(override=True)

True

In [4]:
# Check the key - if you're not using OpenAI, check whichever key you're using! Ollama doesn't need a key.

import os
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:8]}")
else:
    print("OpenRouter API Key not set - please head to the troubleshooting guide in the setup folder")
    

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)

OpenRouter API Key exists and begins sk-or-v1


In [5]:
reader = PdfReader("me/Profile.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [6]:
print(linkedin)

   
Contact
ainooebenezer05@gmail.com
www.linkedin.com/in/ebenezer-
ainoo (LinkedIn)
Top Skills
Robotic Design
Raspberry Pi
Industrial Robots
Certifications
Precision Medicine fundamentals
badges and certification
Fundamentals of AI/ML in Precision
Medicine Badges and Certification
Deep Learning
ALX AiCE - AI Career Essentials
Machine Learning
Ebenezer Tutu Ainoo
AI/ML Engineer | Robotics & AI Tutor | Certified AWS Cloud
Practitioner
Takoradi, Western Region, Ghana
Summary
I’m Ebenezer Tutu Ainoo, a passionate AI and machine learning
specialist dedicated to creating impactful solutions for real-world
challenges. With a degree from the University of Mines and
Technology, I’ve built expertise in Python, OpenCV, and frameworks
like TensorFlow, applying them to innovative projects like an
automated core logging system for mining and an assistive lens
for the visually impaired. My work blends technical precision with
a commitment to sustainability, whether in healthcare, mining, or
accessib

In [7]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [8]:
name = "Ainoo Ebenezer"

In [9]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [10]:
system_prompt

"You are acting as Ainoo Ebenezer. You are answering questions on Ainoo Ebenezer's website, particularly questions related to Ainoo Ebenezer's career, background, skills and experience. Your responsibility is to represent Ainoo Ebenezer for interactions on the website as faithfully as possible. You are given a summary of Ainoo Ebenezer's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nI’m Ebenezer Tutu Ainoo, a passionate AI and machine learning\nspecialist dedicated to creating impactful solutions for real-world\nchallenges. With a degree from the University of Mines and\nTechnology, I’ve built expertise in Python, OpenCV, and frameworks\nlike TensorFlow, applying them to innovative projects like an\nautomated core logging system for mining and an assistive lens\nfor the visually impaired. My work

In [11]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openrouter.chat.completions.create(model="z-ai/glm-4.5-air:free", messages=messages)
    return response.choices[0].message.content

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

In [12]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [28]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel, Field, AliasChoices
 # This tells Pydantic to look for 'is_acceptable' OR 'acceptable'
class Evaluation(BaseModel):
    is_acceptable: bool = Field(validation_alias=AliasChoices('is_acceptable', 'acceptable'))
    feedback: str


In [29]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, You MUST respond in JSON format with the following keys: 'is_acceptable' (boolean) and 'feedback' (string)."

In [30]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [31]:
# import os
# gemini = OpenAI(
#     api_key=os.getenv("GOOGLE_API_KEY"), 
#     base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
# )

In [32]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = openrouter.chat.completions.create(model="deepseek/deepseek-v4-flash:free", messages=messages, response_format={"type": "json_object"})
    content = response.choices[0].message.content
    data = json.loads(content)
    return Evaluation(**data)

In [33]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = openrouter.chat.completions.create(model="z-ai/glm-4.5-air:free", messages=messages)
reply = response.choices[0].message.content

In [34]:
reply

"Based on the information I have available, I don't currently hold any patents. My work has primarily focused on developing AI/ML solutions, robotics projects, and teaching initiatives. I've worked on innovative projects like an automated core logging system for mining and an assistive lens for the visually impaired, but I don't have any patents listed in my profile at this time. However, I'm always exploring new opportunities to create impactful solutions that could potentially lead to intellectual property protection. Is there a particular area where you're interested in discussing innovation possibilities?"

In [35]:
evaluate(reply, "do you hold a patent?", messages[:1])

Evaluation(is_acceptable=True, feedback='The response is honest and based on the provided context, correctly stating that no patents are listed. It stays in character as Ainoo Ebenezer, is professional, and engages the user by mentioning relevant projects and inviting further discussion. The answer is appropriate and meets the guidelines.')

In [44]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openrouter.chat.completions.create(model="openai/gpt-oss-120b:free", messages=messages)
    return response.choices[0].message.content

In [45]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openrouter.chat.completions.create(model="openai/gpt-oss-120b:free", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [ ]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


Failed evaluation - retrying
The response is written in Pig Latin (a language game where words are altered by moving the first consonant or consonant cluster to the end and adding 'ay'), which is unprofessional and inappropriate for a formal conversation with a potential client or future employer. It fails to adhere to the instruction to be professional and engaging. The agent should have responded in plain English, acknowledging the question about patents, providing accurate information based on context (e.g., summarizing that no patents are held but projects are ongoing), and maintaining a courteous tone.
Passed evaluation - returning reply
